# Model Comparison

Goal of this notebook: pick a single best regressor that we will fine-tune in notebook 04. We train seven candidates on the engineered features, run 3-fold cross-validation on each, and rank them by test RMSE.

## Why these seven models?

We cover the main regression families so that whichever wins gives us a clear signal about the structure of the data:

| Family | Models | What it tells us if it wins |
|---|---|---|
| Linear (no regularization) | LinearRegression | The relationship is mostly linear in the engineered features. |
| Linear (regularized) | Ridge, Lasso | Same as above but with shrinkage helping against overfit / collinearity. |
| Instance-based | KNN | Local patterns dominate. |
| Tree ensemble (bagging) | RandomForest | Non-linear interactions matter. |
| Tree ensemble (boosting) | GradientBoosting, XGBoost | Sequential boosting captures stronger non-linearities. |

## Why not SVR?

RBF SVR is O(n^2) on roughly 28k training rows, so it dominates total runtime. On smaller subsamples it lands at the same RMSE as the linear models on this dataset, so we excluded it for time.

## Imports

In [1]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
PROC_DIR = ROOT / 'data' / 'processed'
MODEL_DIR = ROOT / 'models'
FIG_DIR = ROOT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Load processed splits

In [2]:
X_train = pd.read_csv(PROC_DIR / 'X_train_filtered.csv')
X_test = pd.read_csv(PROC_DIR / 'X_test_filtered.csv')
y_train = pd.read_csv(PROC_DIR / 'y_train.csv').squeeze('columns')
y_test = pd.read_csv(PROC_DIR / 'y_test.csv').squeeze('columns')
preprocessor = joblib.load(MODEL_DIR / 'preprocessor.joblib')
print(f'Train {X_train.shape}, Test {X_test.shape}')

Train (28800, 61), Test (7200, 61)
